# Stratégie 3 — Curriculum Learning Progressif (Food101)

Ce notebook implémente une **transition graduelle BF → HF** sur 20 époques, découpées en 4 phases :

| Phase | Époques | Ratio HF | Ratio BF |
|-------|---------|----------|----------|
| BF pur        | 1–5   | 0 %  | 100 % |
| Mixte 25 %    | 6–10  | 25 % | 75 %  |
| Mixte 60 %    | 11–15 | 60 % | 40 %  |
| HF pur        | 16–20 | 100 %| 0 %   |

**Différence vs Stratégie 1** : pas de rupture abrupte BF→HF ; la proportion de HF croît progressivement,  
ce qui devrait lisser l'instabilité observée à la transition dans la Stratégie 1.  
**Loss** : Cross-Entropy standard (pas de pondération per-domaine — différence vs Stratégie 2).

In [ ]:
import os, json, time, random, statistics
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Sampler
from torchvision import datasets, models
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import wandb
    _WANDB_AVAILABLE = True
except ImportError:
    _WANDB_AVAILABLE = False
    print('wandb non installe')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil: {device}')

import sys, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import degradation; importlib.reload(degradation)
from degradation import DegradedDataset, hf_transform, clean_tensor_transform
import cost; importlib.reload(cost)
from cost import data_cost
import env_config; importlib.reload(env_config)
print('Modules charges.')

In [ ]:
DATASET_NAME  = 'Food101'
BASE_DIR      = env_config.ensure_dataset_ready(DATASET_NAME)
RESULTS_DIR   = env_config.results_dir(DATASET_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Dataset    :', DATASET_NAME)
print('BASE_DIR   :', BASE_DIR)
print('RESULTS_DIR:', RESULTS_DIR)

COST_HF        = 10
COST_BF        = 1
BATCH_SIZE     = 64
NUM_WORKERS    = min(8, os.cpu_count() or 2)
TOTAL_EPOCHS   = 20
LR             = 1e-3
HF_TRAIN_RATIO = 0.8
SEEDS          = [42, 1, 2]
USE_WANDB      = True
WANDB_PROJECT  = 'PF22-MultiFidelity'

# (end_epoch_inclusive, hf_ratio_per_batch, phase_label)
CURRICULUM_SCHEDULE = [
    (5,  0.00, 'BF_pur'),
    (10, 0.25, 'Mixte_25pct'),
    (15, 0.60, 'Mixte_60pct'),
    (20, 1.00, 'HF_pur'),
]
PHASE_COLORS = {
    'BF_pur':      '#55A868',
    'Mixte_25pct': '#CCB974',
    'Mixte_60pct': '#E86E55',
    'HF_pur':      '#4C72B0',
}
print('Config curriculum definie.')

In [ ]:
for req in ['train/BF', 'train/HF', 'test']:
    p = os.path.join(BASE_DIR, req)
    if not os.path.isdir(p): raise FileNotFoundError(f'Dossier manquant : {p}')

transform_hf    = hf_transform()
transform_clean = clean_tensor_transform()

dataset_hf_full  = datasets.ImageFolder(os.path.join(BASE_DIR, 'train/HF'), transform=transform_hf)
dataset_bf_train = DegradedDataset(
    datasets.ImageFolder(os.path.join(BASE_DIR, 'train/BF'), transform=transform_clean), seeded=True)
dataset_test_hf  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_hf)
dataset_test_bf  = DegradedDataset(
    datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_clean), seeded=True)

NUM_CLASSES = len(dataset_hf_full.classes)
print(f'Classes ({NUM_CLASSES}): {dataset_hf_full.classes}')
print(f'HF pool: {len(dataset_hf_full)} | BF train: {len(dataset_bf_train)} | Test: {len(dataset_test_hf)}')

In [ ]:
class DomainLabeledConcatDataset(Dataset):
    """Concatenation HF+BF avec etiquette de domaine (1=HF, 0=BF)."""
    def __init__(self, hf_dataset, bf_dataset):
        self.hf = hf_dataset; self.bf = bf_dataset; self.hf_len = len(hf_dataset)
    def __len__(self): return self.hf_len + len(self.bf)
    def __getitem__(self, idx):
        if idx < self.hf_len:
            x, y = self.hf[idx]; return x, y, 1
        x, y = self.bf[idx - self.hf_len]; return x, y, 0

class MixedBatchSampler(Sampler):
    """Force un ratio HF/BF fixe par batch."""
    def __init__(self, hf_indices, bf_indices, batch_size, hf_ratio, seed=42, drop_last=True):
        self.hf_indices   = list(hf_indices); self.bf_indices = list(bf_indices)
        self.batch_size   = batch_size
        self.hf_per_batch = max(1, int(round(batch_size * hf_ratio)))
        self.bf_per_batch = batch_size - self.hf_per_batch
        if self.bf_per_batch <= 0: raise ValueError('hf_ratio trop grand')
        self.seed = seed; self.epoch = 0
        self.num_batches = min(len(self.hf_indices) // self.hf_per_batch,
                               len(self.bf_indices) // self.bf_per_batch)
        if self.num_batches <= 0: raise ValueError('Pas assez de donnees pour un batch mixte.')
    def __len__(self): return self.num_batches
    def set_epoch(self, epoch): self.epoch = epoch
    def __iter__(self):
        g = torch.Generator(); g.manual_seed(self.seed + self.epoch)
        hp = torch.randperm(len(self.hf_indices), generator=g).tolist()
        bp = torch.randperm(len(self.bf_indices),  generator=g).tolist()
        for b in range(self.num_batches):
            batch = ([self.hf_indices[hp[b * self.hf_per_batch + i]] for i in range(self.hf_per_batch)] +
                     [self.bf_indices[bp[b * self.bf_per_batch + j]] for j in range(self.bf_per_batch)])
            order = torch.randperm(len(batch), generator=g).tolist()
            yield [batch[i] for i in order]

def build_curriculum_loaders(hf_train_ds, bf_train_ds, seed=42):
    """Construit un dict {hf_ratio: DataLoader} pour chaque phase du CURRICULUM_SCHEDULE."""
    loaders = {}
    loaders[0.0] = DataLoader(bf_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    loaders[1.0] = DataLoader(hf_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    concat_ds = DomainLabeledConcatDataset(hf_train_ds, bf_train_ds)
    hf_idx = list(range(len(hf_train_ds)))
    bf_idx = list(range(len(hf_train_ds), len(hf_train_ds) + len(bf_train_ds)))
    for _, hf_ratio, _ in CURRICULUM_SCHEDULE:
        if 0.0 < hf_ratio < 1.0 and hf_ratio not in loaders:
            sampler = MixedBatchSampler(hf_idx, bf_idx, BATCH_SIZE, hf_ratio, seed=seed)
            loaders[hf_ratio] = DataLoader(concat_ds, batch_sampler=sampler,
                                            num_workers=NUM_WORKERS, pin_memory=True)
    return loaders

def get_phase(epoch):
    for end_ep, hf_ratio, phase_name in CURRICULUM_SCHEDULE:
        if epoch <= end_ep: return hf_ratio, phase_name
    last = CURRICULUM_SCHEDULE[-1]; return last[1], last[2]

print('Batch samplers et helpers curriculum definis.')

In [ ]:
def create_model(num_classes=10):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(device)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = total_correct = total_n = 0
    with torch.no_grad():
        for batch in loader:
            x, y = batch[0].to(device), batch[1].to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            total_loss    += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n       += x.size(0)
    return total_loss / total_n, 100.0 * total_correct / total_n

def make_grad_scaler():
    try:    return GradScaler('cuda', enabled=(device.type == 'cuda'))
    except TypeError: return GradScaler(enabled=(device.type == 'cuda'))

print('Utilitaires modele definis.')

In [ ]:
def train_curriculum(model, curriculum_loaders, val_loader, n_hf_train, n_bf_train,
                     use_wandb=False):
    """Entrainement sur TOTAL_EPOCHS en suivant CURRICULUM_SCHEDULE.

    Chaque epoque selectionne le loader correspondant a sa phase (ratio HF/BF).
    Loss : Cross-Entropy standard sans ponderation per-domaine.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scaler    = make_grad_scaler()
    history   = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'phase': [], 'hf_ratio': []}
    cumulative_cost = 0
    log_every = 10

    for epoch in range(1, TOTAL_EPOCHS + 1):
        hf_ratio, phase_name = get_phase(epoch)
        loader = curriculum_loaders[hf_ratio]
        if hasattr(loader, 'batch_sampler') and hasattr(loader.batch_sampler, 'set_epoch'):
            loader.batch_sampler.set_epoch(epoch)

        model.train()
        ep_loss = 0.0; ep_n = 0
        ep_start = time.time()

        for bi, batch in enumerate(loader, 1):
            x = batch[0].to(device, non_blocking=True)
            y = batch[1].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            ep_loss += loss.item() * x.size(0); ep_n += x.size(0)
            if bi % log_every == 0 or bi == 1 or bi == len(loader):
                elapsed = time.time() - ep_start
                bps = bi / elapsed if elapsed > 0 else 0.0
                eta = (len(loader) - bi) / bps / 60.0 if bps > 0 else 0.0
                print(f'[{phase_name}] Ep {epoch}/{TOTAL_EPOCHS} | '
                      f'Batch {bi}/{len(loader)} | loss={loss.item():.4f} | '
                      f'{bps:.1f} b/s | ETA={eta:.1f} min')

        train_loss = ep_loss / ep_n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        ep_min = (time.time() - ep_start) / 60.0

        if hf_ratio == 0.0:
            ep_cost = n_bf_train * COST_BF
        elif hf_ratio == 1.0:
            ep_cost = n_hf_train * COST_HF
        else:
            s = loader.batch_sampler
            ep_cost = s.num_batches * (s.hf_per_batch * COST_HF + s.bf_per_batch * COST_BF)
        cumulative_cost += ep_cost

        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc); history['phase'].append(phase_name)
        history['hf_ratio'].append(hf_ratio)

        print(f'[{phase_name}] Ep {epoch}/{TOTAL_EPOCHS} : '
              f'train={train_loss:.4f} | val={val_loss:.4f} | '
              f'val_acc={val_acc:.2f}% | {ep_min:.2f} min')

        if use_wandb and _WANDB_AVAILABLE:
            wandb.log({'epoch': epoch, 'phase': phase_name, 'hf_ratio': hf_ratio,
                       'train/loss': train_loss, 'val/loss': val_loss,
                       'val/accuracy': val_acc, 'cumulative_cost_CA': cumulative_cost,
                       'epoch_time_min': ep_min})

    return history, cumulative_cost

print('Fonction train_curriculum definie.')

In [ ]:
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def run_seed(seed):
    set_all_seeds(seed)
    n_hf     = int(HF_TRAIN_RATIO * len(dataset_hf_full))
    n_hf_val = len(dataset_hf_full) - n_hf
    ds_hf_train, ds_hf_val = random_split(
        dataset_hf_full, [n_hf, n_hf_val],
        generator=torch.Generator().manual_seed(seed))

    ld_val     = DataLoader(ds_hf_val,    batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    ld_test_hf = DataLoader(dataset_test_hf, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    ld_test_bf = DataLoader(dataset_test_bf, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    cl_loaders = build_curriculum_loaders(ds_hf_train, dataset_bf_train, seed=seed)

    track = USE_WANDB and _WANDB_AVAILABLE
    if track:
        wandb.init(project=WANDB_PROJECT,
                   name=f'Strategie3_Curriculum_{DATASET_NAME}_seed{seed}',
                   tags=['strategy3', 'curriculum', DATASET_NAME, f'seed_{seed}'],
                   reinit=True,
                   config={'strategy': 'curriculum_learning', 'dataset': DATASET_NAME,
                           'seed': seed, 'total_epochs': TOTAL_EPOCHS, 'lr': LR,
                           'batch_size': BATCH_SIZE, 'architecture': 'resnet18',
                           'schedule': str(CURRICULUM_SCHEDULE)})

    model = create_model(num_classes=NUM_CLASSES)
    t0 = time.time()
    history, total_cost = train_curriculum(
        model, cl_loaders, ld_val,
        n_hf_train=n_hf, n_bf_train=len(dataset_bf_train),
        use_wandb=track)
    tmin = (time.time() - t0) / 60.0

    crit = nn.CrossEntropyLoss()
    _, hf_acc  = evaluate(model, ld_test_hf, crit)
    _, bf_acc  = evaluate(model, ld_test_bf, crit)
    mix_acc    = (hf_acc + bf_acc) / 2.0

    if track:
        wandb.log({'test/accuracy_HF': hf_acc, 'test/accuracy_BF': bf_acc,
                   'test/accuracy_Mixte': mix_acc})
        wandb.finish()
    print(f'[seed {seed}] HF={hf_acc:.2f}% BF={bf_acc:.2f}% Mixte={mix_acc:.2f}% | '
          f'cost={total_cost:.0f} CA | {tmin:.1f} min')
    return {'seed': seed, 'accuracy_HF': hf_acc, 'accuracy_BF': bf_acc,
            'accuracy_Mixte': mix_acc, 'total_cost_CA': float(total_cost),
            'time_min': tmin, 'n_hf_train': n_hf, 'history': history, 'model': model}

per_seed = [run_seed(s) for s in SEEDS]
model    = per_seed[0]['model']
history  = per_seed[0]['history']

In [ ]:
def _agg(key):
    vals = [r[key] for r in per_seed]
    m = statistics.mean(vals); s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    return m, s, vals

hf_m, hf_s, hf_all = _agg('accuracy_HF')
bf_m, bf_s, bf_all = _agg('accuracy_BF')
mx_m, mx_s, mx_all = _agg('accuracy_Mixte')
cost_m, _, _       = _agg('total_cost_CA')
tt_m,  tt_s, _     = _agg('time_min')

n_hf_pool = len(dataset_hf_full); n_bf_pool = len(dataset_bf_train)
data_cost_CA = data_cost(n_hf=n_hf_pool, n_bf=n_bf_pool)

results = {
    'strategy': 'curriculum_learning_progressif',
    'dataset': DATASET_NAME,
    'curriculum_schedule': [(e, r, p) for e, r, p in CURRICULUM_SCHEDULE],
    'total_epochs': TOTAL_EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'seeds': SEEDS,
    'n_hf_pool': n_hf_pool, 'n_bf_pool': n_bf_pool,
    'data_cost_CA': float(data_cost_CA), 'total_cost_CA': cost_m,
    'accuracy_HF':    hf_m, 'accuracy_HF_std':    hf_s, 'accuracy_HF_seeds':    hf_all,
    'accuracy_BF':    bf_m, 'accuracy_BF_std':    bf_s, 'accuracy_BF_seeds':    bf_all,
    'accuracy_Mixte': mx_m, 'accuracy_Mixte_std': mx_s, 'accuracy_Mixte_seeds': mx_all,
    'total_time_min': tt_m, 'total_time_min_std': tt_s,
    'per_seed': [{k: v for k, v in r.items() if k not in ('model', 'history')}
                 for r in per_seed],
}

json_path = os.path.join(RESULTS_DIR, 'results_strategy3_curriculum_learning.json')
pth_path  = os.path.join(RESULTS_DIR, 'model_strategy3_curriculum_learning.pth')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
torch.save(model.state_dict(), pth_path)

print(f'--- RESULTATS ({len(SEEDS)} seeds) - {DATASET_NAME} - Strategie 3 (Curriculum) ---')
print(f'Test HF    : {hf_m:.2f} +/- {hf_s:.2f} %')
print(f'Test BF    : {bf_m:.2f} +/- {bf_s:.2f} %')
print(f'Test Mixte : {mx_m:.2f} +/- {mx_s:.2f} %')
print(f'Cout total : {cost_m:.0f} CA | Cout donnees : {data_cost_CA:.0f} CA')
print('JSON :', json_path)

In [ ]:
x = np.arange(1, TOTAL_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Strategie 3 (Curriculum Learning) - {DATASET_NAME} (seed {SEEDS[0]})', fontsize=14)

for ax in axes:
    prev = 0
    for end_ep, hf_ratio, phase_name in CURRICULUM_SCHEDULE:
        color = PHASE_COLORS.get(phase_name, '#dddddd')
        ax.axvspan(prev + 0.5, end_ep + 0.5, alpha=0.10, color=color)
        if end_ep < TOTAL_EPOCHS:
            ax.axvline(end_ep + 0.5, color='gray', linestyle=':', linewidth=0.8)
        prev = end_ep

axes[0].plot(x, history['train_loss'], marker='o', markersize=4, linewidth=1.8,
             label='Train loss', color='tab:blue')
axes[0].plot(x, history['val_loss'],   marker='o', markersize=4, linewidth=1.8,
             linestyle='--', label='Val loss HF', color='tab:orange')
axes[0].set_title('Evolution de la loss'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss'); axes[0].grid(alpha=0.3)

axes[1].plot(x, history['val_acc'], marker='o', markersize=4, linewidth=1.8,
             color='tab:green', label='Val acc HF')
axes[1].set_title('Precision validation HF'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)'); axes[1].grid(alpha=0.3)

phase_patches = [mpatches.Patch(color=PHASE_COLORS[p], alpha=0.5, label=p)
                 for _, _, p in CURRICULUM_SCHEDULE]
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + phase_patches, fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95])
png_path = os.path.join(RESULTS_DIR, 'strategy3_curriculum_curves.png')
plt.savefig(png_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figure :', png_path)

In [ ]:
comparison_files = {
    'BL 1(HF)':             'results_baseline_HF.json',
    'BL 2(BF)':             'results_baseline_BF.json',
    'BL 3(MIXTE)':          'results_baseline_MIXTE.json',
    'Strat 1(BF->HF)':      'results_strategy1_transfer_learning.json',
    'Strat 2(CoTrain)':     'results_strategy2_cotraining_reweighting.json',
    'Strat 3(Curriculum)':  'results_strategy3_curriculum_learning.json',
}
palette = {
    'BL 1(HF)':            '#4C72B0', 'BL 2(BF)':             '#55A868',
    'BL 3(MIXTE)':         '#C44E52', 'Strat 1(BF->HF)':      '#8172B2',
    'Strat 2(CoTrain)':    '#CCB974', 'Strat 3(Curriculum)':  '#64B5CD',
}

def _time_min(d):
    for k in ('training_time_min', 'total_time_min'):
        if k in d: return float(d[k])
    if 'training_time_sec' in d: return d['training_time_sec'] / 60.0
    return float('nan')

loaded = {}
for name, fn in comparison_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f: loaded[name] = json.load(f)

if not loaded:
    print('Aucun JSON dans', RESULTS_DIR)
else:
    names     = list(loaded.keys())
    acc_hf    = [loaded[n].get('accuracy_HF',    float('nan')) for n in names]
    acc_bf    = [loaded[n].get('accuracy_BF',    float('nan')) for n in names]
    acc_mixte = [loaded[n].get('accuracy_Mixte', float('nan')) for n in names]
    costs     = [loaded[n].get('total_cost_CA',  float('nan')) for n in names]
    times     = [_time_min(loaded[n]) for n in names]
    colors    = [palette.get(n, '#888888') for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Comparaison — {DATASET_NAME}', fontsize=14)

    x = np.arange(3); width = 0.12; n = len(names)
    offsets = (np.arange(n) - (n - 1) / 2.0) * width
    for i, name in enumerate(names):
        vals = [acc_hf[i], acc_bf[i], acc_mixte[i]]
        bars = axes[0].bar(x + offsets[i], vals, width=width,
                           label=name, color=palette.get(name, '#888888'))
        for bar in bars:
            h = bar.get_height()
            if h == h:
                axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                             f'{h:.1f}', ha='center', va='bottom', fontsize=7)
    axes[0].set_title('Precisions sur les jeux de test'); axes[0].set_xticks(x)
    axes[0].set_xticklabels(['Test HF', 'Test BF', 'Test Mixte'])
    axes[0].set_ylabel('Precision (%)'); axes[0].set_ylim(0, 100)
    axes[0].grid(axis='y', alpha=0.3); axes[0].legend(fontsize=7, ncol=2)

    bars = axes[1].bar(names, costs, color=colors)
    for bar, v in zip(bars, costs):
        if v == v:
            axes[1].text(bar.get_x() + bar.get_width() / 2, v,
                         f"{int(v):,} CA".replace(',', ' '), ha='center', va='bottom', fontsize=8)
    axes[1].set_title('Cout total estime'); axes[1].set_ylabel('CA')
    vc = [c for c in costs if c == c]
    if vc: axes[1].set_ylim(0, max(vc) * 1.15)
    axes[1].grid(axis='y', alpha=0.3); axes[1].tick_params(axis='x', labelrotation=20)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig_path = os.path.join(RESULTS_DIR, 'comparison_all_models_curriculum.png')
    plt.savefig(fig_path, dpi=180, bbox_inches='tight')
    plt.show()

    print(f'\n=== RECAPITULATIF {DATASET_NAME} ===')
    hdr = f"{'Modele':<26} {'HF%':>7} {'BF%':>7} {'Mixte%':>9} {'Cout(CA)':>11} {'Temps(min)':>11}"
    print(hdr); print('-' * len(hdr))
    for i, name in enumerate(names):
        c = int(costs[i]) if costs[i] == costs[i] else 0
        t = times[i] if times[i] == times[i] else 0.0
        print(f"{name:<26} {acc_hf[i]:>7.2f} {acc_bf[i]:>7.2f} {acc_mixte[i]:>9.2f} "
              f"{c:>11} {t:>11.2f}")